# Лабораторная работа №1: Transfer learning и fine-tuning визуального backbone

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
- Сравнить frozen, partial и full fine-tuning ResNet18.
- Построить confusion matrix и проверить corruption shift.


## Теоретическая часть

Предобученный backbone реализует отображение $f_\theta(x)=z$. Для нового набора классов заменяется головка $g_\phi(z)$; сначала оптимизируется только $\phi$, затем часть или все параметры $\theta$. Чем меньше данных, тем выше риск разрушить полезное представление большим learning rate.

![Пример изображений из руководства PyTorch по transfer learning](https://docs.pytorch.org/tutorials/_images/sphx_glr_transfer_learning_tutorial_001.png)

Нужно сравнивать не только accuracy, но и macro-F1, время, память и устойчивость к domain shift.


### Математическая постановка

Для многоклассовой задачи используется cross-entropy:
$$\mathcal L=-\frac1N\sum_{i=1}^{N}\log p_\theta(y_i\mid x_i).$$
Три режима эксперимента: frozen backbone, размороженный последний блок, полный fine-tuning.


### Материалы

- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [torchvision models](https://pytorch.org/vision/stable/models.html)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
2. Сравнить frozen, partial и full fine-tuning ResNet18.
3. Построить confusion matrix и проверить corruption shift.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** ResNet18, FP32/FP16, 32–224 px.
- **Ограничение данных:** до 10 000 train-изображений в обязательном запуске.
- **Fallback:** CPU/FAST_DEV_RUN на подвыборке.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


## Реализация


In [ ]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = True  # False — полный эксперимент
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


In [ ]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Загрузка CIFAR-10 и воспроизводимые преобразования

**Постановка.** Разделение подготовки данных от моделирования — базовое требование к воспроизводимым экспериментам: детерминированная загрузка, фиксированные преобразования и контролируемая генерация батчей позволяют атрибутировать изменение метрик изменениям модели, а не случайным различиям пайплайна.

Выполните следующие действия:

1. Загрузите CIFAR-10 через torchvision (train и test), зафиксируйте `SEED = 42`, `DATA_ROOT` и каталог `ARTIFACTS`.
2. Постройте transforms: train — `Resize(96)`, `RandomHorizontalFlip`, `ToTensor`; test — только `Resize(96)` и `ToTensor` (без аугментаций).
3. Соберите `DataLoader`: train — c перемешиванием, test — детерминированный; выведите размеры выборок, счётчик классов и форму/диапазон одного пример-тензора.
4. Убедитесь, что воспроизводимость подтверждается: повторный запуск ячейки даёт идентичный порядок элементов и то же самое контрольное изображение.

**Результат.** Сигнатуры `train_ds/test_ds/train_loader/test_loader` в ячейке вывода; счётчики классов.

**Критерий принятия.** Задание считается принятым, если загрузчик отдаёт батчи формы (B, 3, 96, 96); счётчики классов распечатаны; повторный запуск не меняет результат.

In [ ]:
# Реализуйте это задание по постановке выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Модель и цикл обучения для переноса знаний

**Постановка.** Реализация, полученная в настоящем задании, образует вычислительное ядро практикума: от её корректности зависит валидность всех последующих измерений.

Выполните следующие действия:

1. Реализуйте необходимые функции: `build_model(mode, num_classes)`, `train_epoch(model, loader, optimizer)`, `evaluate(model, loader)`.
2. Выполните последовательность вычислительных шагов (пять итераций); все случайные величины фиксируйте через ранее объявленный `SEED`.
3. Выведите итоговые значения в ячейке вывода отдельной инструкцией вывода — эти значения используются в последующих заданиях.
4. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода; значения используются в последующих заданиях.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните сигнатуры и формат артефактов, описанные выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Сравнение режимов fine-tuning по качеству и времени

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Для каждого режима выполните одинаковое число обучающих шагов: 1 при `FAST_DEV_RUN = True`, 5 при полном прогоне — так сравнение честное.
2. Задайте learning rates по типу весов: 3e-4 для новой головы, 3e-5 для размороженных весов backbone.
3. Замерьте wall-clock время каждого режима (`time.perf_counter`) и соберите итог: `mode → (accuracy, f1, seconds)`.
4. Выведите сводную таблицу так, чтобы разницу режимов было видно сразу; прокомментируйте компромисс «качество/время».

**Результат.** `artifacts/metrics.json`; таблица режим-→метрики в ячейке вывода.

**Критерий принятия.** Задание считается принятым, если сравнение выполнено при одинаковых данных и шагах; времена и метрики непустые; файл записан.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/metrics.json

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Устойчивость лучшего режима к гауссовскому шуму

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Выберите лучший режим по accuracy — дальше тестируем именно его.
2. Реализуйте обёртку `Noisy(base, sigma)`: к каждому test-тензору добавляется гауссовский шум `N(0, sigma)`, результат обрезается в [0, 1].
3. Вычислите accuracy при `sigma = 0.15` и сравните с чистой точностью того же режима.
4. Убедитесь, что падение не катастрофично (разница accuracy < 0.2); otherwise — ослабьте шум или верните шаг раньше.

**Результат.** `artifacts/metrics.json` дополнен ключом `shift`: пара значений «чистая vs шумная» accuracy.

**Критерий принятия.** Задание считается принятым, если падение accuracy посчитано и сохранено; вывод об устойчивости сделан явно.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/metrics.json

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Оценка на domain shift

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Выберите лучший режим по accuracy — дальше тестируем именно его.
2. Реализуйте обёртку `Noisy(base, sigma)`: к каждому test-тензору добавляется гауссовский шум `N(0, sigma)`, результат обрезается в [0, 1].
3. Вычислите accuracy при `sigma = 0.15` и сравните с чистой точностью того же режима.
4. Убедитесь, что падение не катастрофично (разница accuracy < 0.2); otherwise — ослабьте шум или верните шаг раньше.

**Результат.** `artifacts/metrics.json` дополнен ключом `shift`: пара значений «чистая vs шумная» accuracy.

**Критерий принятия.** Задание считается принятым, если падение accuracy посчитано и сохранено; вывод об устойчивости сделан явно.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/metrics.json

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Число обучаемых параметров в режимах frozen, partial и full

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Для каждого режима build_model("frozen"/"partial"/"full") посчитайте число обучаемых и всех параметров.

**Результат.** Убедитесь, что у frozen обучаемых меньше, чем у partial, а у partial — меньше, чем у full. Сохраните artifacts/param_report.json.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Проверьте: у frozen обучаемых меньше, чем у partial, а у partial — меньше, чем у full. Сохраните artifacts/param_report.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Точность по классам и худший класс

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Из results["full"]["cm"] вычислите accuracy по каждому из 10 классов; найдите худший класс.

**Результат.** Сохраните artifacts/class_report.json.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните artifacts/class_report.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Эффективность режимов: точность за секунду

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. На основе results постройте список {mode, seconds, accuracy} и определите режим с лучшим accuracy per second.

**Результат.** Сохраните artifacts/efficiency_report.json.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните artifacts/efficiency_report.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Сводка запуска: режимы, шум и ошибки по классам

*Вес: 0.5 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Соберите в один artifacts/run_summary.json: seed, режимы с их accuracy/seconds, sigma-свип, худший класс.
2. Выполните assert, что все обязательные ключи присутствуют.

**Критерий принятия.** Задание считается принятым, если выполните assert, что все обязательные ключи присутствуют, результат виден в ячейке вывода в этом ноутбуке и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

In [ ]:
# Самопроверка: все обязательные ячейки выше должны быть исполнены (кроме бонусной), в outputs не должно быть ошибок.
# Если какая-то ячейка не исполнена или упала - вернитесь и доведите её до конца.
check(True, "самопроверка: обязательные ячейки исполнены, ошибок нет")

### Инженерия результатов

Ниже — усложнённые задания обязательной части. Они не повторяют базовые, а достраивают вокруг результата лабы инженерный контур: стресс-тест, статистическая значимость, воспроизводимость, стоимость и сводный дашборд. Выполняются по порядку: последнее собирает итог из предыдущих. Вес каждого — 1 балл.

### Стресс-тест: качество при возмущении входа

**Вес: 1 балл, обязательное.** Возьмите основной результат лабы как «счётчик качества» и постройте кривую его деградации при нарастающих возмущениях (несколько уровней, повторы на каждый). Ключевой вопрос — порог: при какой силе возмущения результат перестаёт быть пригодным. Артефакт: `artifacts/stress_robustness.json`.

In [ ]:
# Контекст лабы: lab 01 transfer learning. Счётчик качества — число из результатов этой лабы (метрика, IoU, accuracy, качественная оценка).
# Что нужно сделать:
# 1) определить количественный счётчик качества по результатам лабы;
# 2) построить его зависимость от силы возмущения входа (3-5 уровней, по 3 повтора на уровень);
# 3) записать кривую (сила, среднее, std) и относительную деградацию;
# 4) вывод: при какой силе возмущения результат ещё приемлем.
# Артефакт: artifacts/stress_robustness.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Доверительные интервалы: бутстреп и значимость разницы

**Вес: 1 балл, обязательное.** Одного среднего недостаточно: превратите результат в выборку испытаний, постройте 95% бутстреп-интервал для основной конфигурации и для ослабленной, проверьте, пересекаются ли интервалы. Отдельно посмотрите, как ширина интервала зависит от числа ресэплов. Артефакт: `artifacts/bootstrap_ci.json`.

In [ ]:
# Контекст лабы: lab 01 transfer learning.
# Что нужно сделать:
# 1) превратить основной результат в выборку испытаний (успех/неудача) или числовых значений;
# 2) бутстреп (>=400 ресэплов) 95% CI для основной и для ослабленной конфигурации (например, x0.9);
# 3) пересекаются ли интервалы - вывод о значимости разницы;
# 4) бонус-наблюдение: как ширина CI зависит от числа ресэплов (100/400/1600).
# Артефакт: artifacts/bootstrap_ci.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Воспроизводимость: разброс по случайным зернам (seed)

**Вес: 1 балл, обязательное.** Повторите ключевой эксперимент лабы при нескольких состояниях генератора случайных чисел (если полный запуск дорогой — на подвыборке), оцените стандартное отклонение и ответьте: можно ли доверять одиночному запуску. Артефакт: `artifacts/seed_variance.json`.

In [ ]:
# Контекст лабы: lab 01 transfer learning.
# Что нужно сделать:
# 1) повторить ключевой эксперимент лабы при 4 разных seed (если дорого - на подвыборке);
# 2) записать пары (seed, score) и посчитать стандартное отклонение;
# 3) вывод: можно ли доверять одиночному запуску; что конкретно улучшит воспроизводимость (фиксация генераторов, детерминизм,фиксация генераторов...);
# 4) отдельно: какие операции в лабе недетерминированы.
# Артефакт: artifacts/seed_variance.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Стоимость: время, память и вес артефактов

**Вес: 1 балл, обязательное.** Замерьте, во что обходится результат: время ключевого прогона, пик памяти (tracemalloc), суммарный вес артефактов. Найдите главный вкладчик и постройте оценку, как стоимость растёт при десятикратном увеличении данных. Артефакт: `artifacts/cost_profile.json`.

In [ ]:
# Контекст лабы: lab 01 transfer learning.
# Что нужно сделать:
# 1) замерить время ключевого прогона, пик памяти (tracemalloc) и суммарный вес артефактов;
# 2) найти самого тяжёлого вкладчика (по времени/памяти/размеру);
# 3) прогноз: как стоимость растёт при увеличении данных в 10 раз (по какому компоненту - линейно, квадратично);
# 4) вывод: главное вычислительное узкое место лабы.
# Артефакт: artifacts/cost_profile.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Сводный дашборд: все проверки лабы в одном отчёте

**Вес: 1 балл, обязательное.** Соберите результаты всех предыдущих проверок и артефакты лабы в один отчёт: словарь флагов PASS/WARN/MISSING по каждому направлению, машиночитаемый JSON и человекочитаемый markdown. Это итоговая самодиагностика лабораторной. Артефакты: `artifacts/final_dashboard.json`, `artifacts/final_dashboard.md`.

In [ ]:
# Контекст лабы: lab 01 transfer learning. Задание собирает итог из предыдущих - выполняется последним.
# Что нужно сделать:
# 1) прочитать json-результаты всех проверок выше (и остальные артефакты лабы);
# 2) выставить по каждому направлению флаг PASS/WARN/MISSING с порогами, которые вы объявите явно;
# 3) сохранить artifacts/final_dashboard.json (машиночитаемый) и artifacts/final_dashboard.md (человекочитаемый);
# 4) резюме лабы в трёх строках: что работает, что осторожно, что не проверено.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит файлов артефактов и их размеров

*Вес: 0.5 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Соберите список всех файлов в каталоге artifacts/, для каждого — размер в байтах; убедитесь, что ни один не пуст.

**Результат.** Сохраните artifacts/files_audit.json.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните artifacts/files_audit.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Мини-исследование: гипотеза, эксперимент и вывод

*Вес: 0.5 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Гипотеза.
2. Эксперимент с фиксированным seed.
3. Метрика и сравнение.
4. Вывод: подтвердилась/нет, почему.

**Критерий принятия.** Задание считается принятым, если вывод: подтвердилась/нет, почему, результат виден в ячейке вывода в этом ноутбуке и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

## Анализ результатов

Заполните после запуска:

1. Таблица сравниваемых конфигураций и вычислительный бюджет.
2. Основные метрики с доверительными интервалами или вариативностью по seed, где это возможно.
3. Не менее пяти характерных ошибок/неудачных примеров.
4. Ablation: какой компонент действительно дал эффект?
5. Ограничения эксперимента и следующий проверяемый шаг.


In [ ]:
# Автоматическая проверка наличия каталога артефактов
assert ARTIFACTS.exists()
print("Artifacts:", [p.name for p in ARTIFACTS.iterdir()])


## Выводы

Сформулируйте 3–5 выводов, каждый из которых опирается на измерение или наблюдаемый пример. Не подменяйте вывод перечислением выполненных действий.
